Copyright **`(c)`** 2026 Giovanni Squillero `<squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free for personal or classroom use; see 'LICENCE.md' for details.

In [1]:
import random
from itertools import permutations, product
import numpy as np
from tqdm.notebook import tqdm

In [ ]:
TICTACTOE_MAP = np.array([[1, 6, 5], [8, 4, 0], [3, 2, 7]])


def display(state, legend=None, *, coordinates=False):
    if not legend:
        legend = list()
    legend += [''] * 3
    x, o = state
    for r, c in product(range(3), repeat=2):
        if TICTACTOE_MAP[r, c] in x:
            print("X", end=" ")
        elif TICTACTOE_MAP[r, c] in o:
            print("O", end=" ")
        elif coordinates:
            print(f"{TICTACTOE_MAP[r, c]}", end=" ")
        else:
            print(".", end=" ")
        if c == 2:
            print(f" {legend[r]}")

In [ ]:
def winning_position(cells):
    return any(sum(h) == 12 for h in permutations(cells, 3))


def eval_static(state: tuple[frozenset[int], frozenset[int]]) -> int:
    """Statically evaluate a board: 1 if agent won, -1 if it lost"""
    if winning_position(state[0]):
        return 1
    elif winning_position(state[1]):
        return -1
    else:
        return 0

In [ ]:
def next_state(state: tuple[frozenset[int], frozenset[int]], action: int) -> tuple[frozenset[int], frozenset[int]]:
    """Returns the next state when agent does `action` in `state`"""
    me, opponent = state
    assert len(me) <= len(opponent)
    return frozenset(set(me) | {action}), opponent


def valid_actions(state: tuple[frozenset[int], frozenset[int]], agent: int = 0) -> list[int]:
    """Returns a list of valid actions"""
    if len(state[agent]) > len(state[1 - agent]):
        return list()
    else:
        return list(set(range(9)) - state[0] - state[1])


def best_action(Q: dict, state: tuple[frozenset[int], frozenset[int]]):
    if not valid_actions(state):
        return (None, eval_static(state))
    else:
        return max(((a, Q[(state, a)]) for a in valid_actions(state)), key=lambda x: x[1])

In [ ]:
def describe_policy(Q, V):
    non_zero = [q for q in Q.items() if q[1] != 0]
    learned = [q for q in Q.items() if q[1] != 0 and q[1] != 1 and q[1] != -1]
    print(f"Found {len(non_zero):,} non zero s/a over {len(Q):,}; {len(learned):,} learned.")
    non_zero = [v for v in V.items() if v[1] != 0]
    learned = [v for v in V.items() if v[1] != 0 and v[1] != 1 and v[1] != -1]
    print(f"Found {len(non_zero):,} non zero values over {len(V):,}; {len(learned):,} learned.")

In [ ]:
states = set()
states |= set(
    (frozenset(x), frozenset(y))
    for n in range(5)
    for x in permutations(range(9), n)
    for y in permutations(set(range(9)) - set(x), n)
)
states |= set(
    (frozenset(x), frozenset(y))
    for n in range(5)
    for x in permutations(range(9), n+1)
    for y in permutations(set(range(9)) - set(x), n)
)
states |= set(
    (frozenset(x), frozenset(y))
    for n in range(5)
    for x in permutations(range(9), n)
    for y in permutations(set(range(9)) - set(x), n + 1)
)

states_sorted = sorted(states, key=lambda s: len(s[0]) + len(s[1]))
final_states = {s for s in states_sorted if len(s[0])+len(s[1]) == 9 or eval_static(s) != 0}
V_table = {s: eval_static(s) for s in states_sorted}
Q_table = {(s, a): eval_static(next_state(s, a)) for s in states_sorted for a in valid_actions(s)}
print(f"Found {len(states):,} states, {len(final_states):,} marked as final")
print(f"V-table contains {len(V_table):,} state values")
print(f"Q-table contains {len(Q_table):,} state-action pairs (utility values)")

In [ ]:
gamma = 0.9

for epochs in tqdm(range(100)):
    Q_table_next = dict(Q_table)
    for s, a in Q_table:
        sn = next_state(s, a)
        if sn in final_states:
            q = V_table[sn]
        else:
            next_moves = [best_action(Q_table, (sn[0], frozenset(set(sn[1])|{x}))) for x in valid_actions(sn, agent=1)]
            if not valid_actions(sn, agent=1):
                display(sn)
                print(sn in final_states)
            q = sum(v for _, v in next_moves)/len(next_moves)
        Q_table_next[(s, a)] = V_table[s] + gamma * q
    Q_table = Q_table_next
describe_policy(Q_table, V_table)

In [ ]:
board = "XOX" \
        "..O" \
        "..."
state = (frozenset(c for i, c in enumerate(TICTACTOE_MAP.flatten()) if board[i] == 'X'), frozenset(c for i, c in enumerate(TICTACTOE_MAP.flatten()) if board[i] == 'O'))
display(state, coordinates=True)
print()

for a in valid_actions(state):
    display((state[0]|{a}, state[1]), [f'Action: {a}', f'Expected reward: {Q_table[(state, a)]}'])
    print()
